Justificación de la Arquitectura de Almacenamiento (Hito 00)

MongoDB Atlas (NoSQL): Se utiliza como Data Lake para ingestar y unificar las fuentes de noticias (Kaggle, Finnhub, Google News). Al ser datos semiestructurados (texto libre) y provenir de APIs con esquemas variables, MongoDB ofrece la flexibilidad necesaria para unificar los documentos sin un esquema rígido inicial.

AWS RDS (SQL): Se utiliza como Data Warehouse. Tras extraer los datos de MongoDB, aplicar procesamiento de Lenguaje Natural (VADER NLP) y cruzarlos con datos financieros estructurados (Yahoo Finance), el resultado es un dataset tabular estrictamente tipado. RDS garantiza la integridad referencial y temporal necesaria para alimentar el modelo de Machine Learning.

In [1]:
# ==============================================================================
# HITO 0: ORQUESTACIÓN Y PROCESAMIENTO 
# ==============================================================================
import pandas as pd
import numpy as np
import yfinance as yf
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from dotenv import load_dotenv
import os
from datetime import datetime

# Cargar variables de entorno
load_dotenv("../.env")

print("INICIANDO")

# 1. CARGA Y CONSOLIDACIÓN DE NOTICIAS
dfs = []
archivos = ["../datasets/noticiaskaggle.csv", "../datasets/noticiasfinnhubNVDA.csv", "../datasets/noticiasGoogleNVDA.csv"]

for f in archivos:
    if os.path.exists(f):
        df_temp = pd.read_csv(f, on_bad_lines='skip')
        if 'headline' in df_temp.columns: df_temp = df_temp.rename(columns={'headline': 'title'})
        # Aseguramos que tenemos las columnas necesarias
        if 'date' in df_temp.columns and 'title' in df_temp.columns:
            df_temp = df_temp[['date', 'title']]
            dfs.append(df_temp)

df_noticias = pd.concat(dfs, ignore_index=True)

# CORRECCIÓN DE FECHAS ROBUSTA:
# 1.1. Convertimos a datetime con utc=True para estandarizar
df_noticias['date'] = pd.to_datetime(df_noticias['date'], format='mixed', errors='coerce', utc=True)

# 1.2. Eliminamos filas inválidas
df_noticias = df_noticias.dropna(subset=['date', 'title'])

# 1.3. Quitamos la zona horaria (hacemos el datetime "naive") y normalizamos
df_noticias['date'] = df_noticias['date'].dt.tz_localize(None).dt.normalize()

# 2. FILTRADO SECTORIAL
keywords = ['nvda', 'nvidia', 'semiconductor', 'chip', 'amd', 'intel', 'tsmc', 'gpu', 'ai']
pattern = '|'.join([f"\\b{kw}\\b" for kw in keywords])
df_noticias = df_noticias[df_noticias['title'].str.contains(pattern, case=False, na=False)].copy()

# 3. NLP Y MEMORIA DE MERCADO (EMA)
print("Aplicando NLP y Memoria de Mercado...")
analyzer = SentimentIntensityAnalyzer()
df_noticias['sentiment_raw'] = df_noticias['title'].apply(lambda x: analyzer.polarity_scores(str(x))['compound'])

# Agrupar por fecha
sentimiento_diario = df_noticias.groupby('date').agg({'sentiment_raw': 'mean', 'title': 'count'})
sentimiento_diario.columns = ['sentiment_score_daily', 'news_volume']

# Reindexar con frecuencia de días laborables
idx = pd.date_range(start=sentimiento_diario.index.min(), end=datetime.now().date(), freq='B')
sentimiento_diario = sentimiento_diario.reindex(idx).ffill(limit=3).fillna(0)
# Aplicar EMA (3 días)
sentimiento_diario['sentiment_score'] = sentimiento_diario['sentiment_score_daily'].ewm(span=3, adjust=False).mean()

# 4. DESCARGA TÉCNICA (YAHOO FINANCE)
print("Descargando precios...")
df_precios = yf.download("NVDA", start=sentimiento_diario.index.min(), progress=False)

# --- CORRECCIÓN NUCLEARES DE COLUMNAS ---
# 1. Si es MultiIndex, aplanamos las columnas quedándonos solo con el nombre del precio
if isinstance(df_precios.columns, pd.MultiIndex):
    df_precios.columns = df_precios.columns.get_level_values(0)

# 2. Forzamos nombres estándar (a veces yfinance descarga 'Adj Close' en lugar de 'Close')
# Si el DataFrame no tiene 'Close', intentamos usar 'Adj Close' renombrándolo
if 'Close' not in df_precios.columns and 'Adj Close' in df_precios.columns:
    df_precios = df_precios.rename(columns={'Adj Close': 'Close'})

# Imprimimos columnas para verificar qué tenemos (esto nos salvará la vida si falla)
print(f"DEBUG: Columnas detectadas: {df_precios.columns.tolist()}")

# Asegurar que el índice sea Datetime normalizado
df_precios.index = pd.to_datetime(df_precios.index).normalize()
# --------------------------------------

# Cálculo RSI (usando la columna 'Close' recién asegurada)
delta = df_precios['Close'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
df_precios['rsi_14'] = 100 - (100 / (1 + (gain / loss)))

# 5. FUSIÓN FINAL
print("Fusionando y calculando variable objetivo...")
# Aseguramos que tenemos las columnas necesarias antes del join
df_master = df_precios.join(sentimiento_diario[['sentiment_score', 'news_volume']], how='left').fillna(0)

# APLICACIÓN DIRECTA DE TARGET_RETURN (Verificamos que 'Close' existe)
if 'Close' in df_master.columns:
    df_master['Target_Return'] = df_master['Close'].pct_change().shift(-1)
else:
    print("ERROR: La columna 'Close' no existe en df_master. Revisa las columnas: ", df_master.columns)

# Limpieza final
df_master.dropna(inplace=True)

# 6. GUARDADO Y VERIFICACIÓN
df_master.to_csv("../datasets/dataset_entrenamiento_final.csv")
print(f"¡Dataset Maestro consolidado! Filas totales: {len(df_master)}")

# VERIFICACIÓN DE COLUMNAS ANTES DE IMPRIMIR
print("Columnas disponibles: ", df_master.columns.tolist())
print(df_master[['Close', 'sentiment_score', 'Target_Return']].tail())


# 7. INYECCIÓN EN AWS RDS
print("Iniciando inyección en AWS RDS...")

try:
    # Construcción del string de conexión (usando variables de tu .env)
    # Asegúrate de tener estas variables definidas en tu archivo .env
    DB_USER = os.getenv('DB_USER')
    DB_PASSWORD = os.getenv('DB_PASSWORD')
    DB_HOST = os.getenv('DB_HOST')
    DB_PORT = os.getenv('DB_PORT')
    DB_NAME = os.getenv('DB_NAME')
    
    conexion_str = f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(conexion_str)
    
    # Subir a la base de datos (reemplazando la tabla existente para mantenerla fresca)
    df_master.to_sql(
        name='dataset_nvda_maestro',
        con=engine, 
        if_exists='replace', 
        index=True # Mantenemos la fecha como índice en la BD
    )
    
    print(f"¡ÉXITO! Dataset Maestro inyectado en AWS RDS.")
    print("Ahora tu modelo puede consultar directamente esta tabla en la nube.")
    
except Exception as e:
    print(f" Error al subir a RDS: {e}")
    

INICIANDO
Aplicando NLP y Memoria de Mercado...
Descargando precios...
DEBUG: Columnas detectadas: ['Close', 'High', 'Low', 'Open', 'Volume']
Fusionando y calculando variable objetivo...
¡Dataset Maestro consolidado! Filas totales: 4241
Columnas disponibles:  ['Close', 'High', 'Low', 'Open', 'Volume', 'rsi_14', 'sentiment_score', 'news_volume', 'Target_Return']
                 Close  sentiment_score  Target_Return
Date                                                  
2026-05-29  210.894211         0.109519       0.062612
2026-06-01  224.098816         0.122759      -0.006864
2026-06-02  222.560608         0.157396      -0.036218
2026-06-03  214.500000         0.143250       0.019394
2026-06-04  218.660004         0.118166      -0.062014
Iniciando inyección en AWS RDS...
 Error al subir a RDS: name 'create_engine' is not defined


### Explicacion sobre el código:
Este código realiza la ingesta, procesamiento y orquestación de datos para crear un dataset maestro
que combina noticias financieras con datos de precios de NVDA. El proceso incluye:
1. Carga y consolidación de noticias desde múltiples fuentes.
2. Filtrado sectorial para enfocarnos solo en noticias relevantes a NVDA y semiconductores.
3. Aplicación de NLP (VADER) para obtener un score de sentimiento diario, suavizado con EMA.
4. Descarga de precios históricos de NVDA desde Yahoo Finance.
5. Cálculo de indicadores técnicos (RSI) y fusión con el sentimiento.
6. Cálculo de la variable objetivo (Target_Return) y limpieza final.
7. Inyección del dataset final en AWS RDS para su uso en la nube.